In [1]:
!pip install -q datasets huggingface_hub transformers

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_token")
login(token=hf_token)

In [4]:
import pandas as pd
import kagglehub
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, mixed_precision
from datasets import load_dataset
import multiprocessing as mp
import numba
import numba.core.types as nb_types
from numba.typed import Dict
mixed_precision.set_global_policy("mixed_float16")

In [4]:
dataset = load_dataset("mateuszgrzyb/lichess-stockfish-normalized", split="train[:95000000]")

README.md: 0.00B [00:00, ?B/s]

train-00000.parquet:   0%|          | 0.00/726M [00:00<?, ?B/s]

train-00001.parquet:   0%|          | 0.00/728M [00:00<?, ?B/s]

train-00002.parquet:   0%|          | 0.00/707M [00:00<?, ?B/s]

train-00003.parquet:   0%|          | 0.00/668M [00:00<?, ?B/s]

...

train-00006.parquet:   0%|          | 0.00/633M [00:00<?, ?B/s]

train-00007.parquet:   0%|          | 0.00/684M [00:00<?, ?B/s]

train-00008.parquet:   0%|          | 0.00/717M [00:00<?, ?B/s]

train-00009.parquet:   0%|          | 0.00/552M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/316072343 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['fen', 'depth', 'cp', 'mate'],
    num_rows: 95000000
})

In [6]:
def keep_reasonable(batch):
    return [
        s is not None and -2500 <= s <= 2500
        for s in batch["cp"]
    ]

dataset = dataset.filter(
    keep_reasonable,
    batched=True,
    batch_size=10000,
    num_proc=os.cpu_count(),  
                               
)

Filter (num_proc=4):   0%|          | 0/95000000 [00:00<?, ? examples/s]

In [7]:
dataset


Dataset({
    features: ['fen', 'depth', 'cp', 'mate'],
    num_rows: 84343546
})

In [8]:
# --- POV sanity check: is `cp` White-relative or side-to-move-relative? ---
#
# This dataset's `fen`/`cp`/`depth`/`knodes` schema is a direct pass-through
# of Lichess's raw evaluation dump (lichess_db_eval.jsonl), i.e. raw UCI
# engine output. By UCI convention, engine scores are relative to the side
# to move -- NOT always White. (The "always White POV" convention people
# often remember is for the *separate* PGN `%eval` annotations Lichess
# embeds in game comments, which is a different, unrelated dataset/pipeline.)
#
# Rather than take that on faith, we verify it empirically on a sample of
# this exact dataset: naive material balance (White material - Black
# material) should correlate POSITIVELY with `cp` when it's White's move.
# If `cp` is side-to-move-relative, that correlation should FLIP SIGN
# (become negative) when it's Black's move. If `cp` were always White-POV,
# the correlation sign would stay the same regardless of whose turn it is.

PIECE_VALUES = {'p': 1, 'n': 3, 'b': 3, 'r': 5, 'q': 9}

def material_balance(fen):
    placement = fen.split()[0]
    bal = 0
    for ch in placement:
        low = ch.lower()
        if low in PIECE_VALUES:
            bal += PIECE_VALUES[low] if ch.isupper() else -PIECE_VALUES[low]
    return bal

_sample = dataset.shuffle(seed=0).select(range(min(5000, len(dataset))))
_white_pairs, _black_pairs = [], []
for _row in _sample:
    _fen, _cp = _row["fen"], _row["cp"]
    if _cp is None:
        continue
    _bal = material_balance(_fen)
    if _fen.split()[1] == 'w':
        _white_pairs.append((_bal, _cp))
    else:
        _black_pairs.append((_bal, _cp))

def _corr(pairs):
    if len(pairs) < 2:
        return float('nan')
    xs = np.array([p[0] for p in pairs], dtype=np.float64)
    ys = np.array([p[1] for p in pairs], dtype=np.float64)
    if xs.std() == 0 or ys.std() == 0:
        return float('nan')
    return float(np.corrcoef(xs, ys)[0, 1])

_corr_white = _corr(_white_pairs)
_corr_black = _corr(_black_pairs)
print(f"corr(material_balance, cp) | White to move: {_corr_white:.3f} (n={len(_white_pairs)})")
print(f"corr(material_balance, cp) | Black to move: {_corr_black:.3f} (n={len(_black_pairs)})")

if _corr_white > 0 and _corr_black < 0:
    CP_IS_WHITE_POV = False
    print("=> 'cp' is already relative to the SIDE TO MOVE. No sign flip needed.")
elif _corr_white > 0 and _corr_black > 0:
    CP_IS_WHITE_POV = True
    print("=> 'cp' is relative to WHITE. Will flip sign for Black-to-move positions.")
else:
    CP_IS_WHITE_POV = False
    print("=> Correlation signs inconclusive on this sample (small-sample noise is "
          "possible). Defaulting to side-to-move, which matches the documented "
          "Lichess eval-dump convention -- re-run this cell with a bigger sample "
          "if you want to double check.")

corr(material_balance, cp) | White to move: 0.529 (n=2522)
corr(material_balance, cp) | Black to move: 0.453 (n=2478)
=> 'cp' is relative to WHITE. Will flip sign for Black-to-move positions.


In [10]:
del _sample

In [11]:
import numba.core.types as nb_types
from numba.typed import Dict, List

PIECE_CHAR_TO_TYPE_RAW = {'p': 0, 'n': 1, 'b': 2, 'r': 3, 'q': 4}
PIECE_CHAR_TO_TYPE = Dict.empty(
    key_type=nb_types.unicode_type,
    value_type=nb_types.int64,
)
for k, v in PIECE_CHAR_TO_TYPE_RAW.items():
    PIECE_CHAR_TO_TYPE[k] = v

# Define the type for the pieces list: List[Tuple(int64, int64, bool)]
piece_type = nb_types.Tuple((nb_types.int64, nb_types.int64, nb_types.boolean))

@numba.jit(nopython=True, cache=True)
def parse_fen(fen, piece_map):
    placement, turn = fen.split()[:2]
    pieces = List.empty_list(piece_type)
    king_sq = np.empty(2, dtype=np.int64)
    king_sq[0] = -1
    king_sq[1] = -1

    rank, file = 7, 0
    for ch in placement:
        if ch == '/':
            rank -= 1
            file = 0
        elif '0' <= ch <= '9':
            file += ord(ch) - ord('0')
        else:
            is_white = ch.isupper()
            square = rank * 8 + file
            if ch.lower() == 'k':
                if is_white:
                  king_sq[1] = square
                else:
                  king_sq[0] = square
            else:
                pieces.append((np.int64(square), np.int64(piece_map[ch.lower()]), is_white))
            file += 1

    us_is_white = (turn == 'w')
    return pieces, king_sq, us_is_white

In [12]:
import logging

# Suppress Numba warnings by setting its logger level to ERROR
logging.getLogger('numba').setLevel(logging.ERROR)

In [13]:
@numba.jit(nopython=True, cache=True)
def mirror_sq(square):
    return square ^ 56

@numba.jit(nopython=True, cache=True)
def orient(square, perspective_is_white):
    return square if perspective_is_white else mirror_sq(square)

@numba.jit(nopython=True, cache=True)
def halfkp_from_parsed(pieces, king_sq, perspective_is_white):
    # Extract king square based on perspective
    k_idx = king_sq[1] if perspective_is_white else king_sq[0]
    if k_idx == -1:
        return [np.uint16(0) for _ in range(0)] # Explicitly return empty list of uint16

    k = orient(k_idx, perspective_is_white)
    indices = []
    for square, ptype, is_white in pieces:
        relative = 0 if is_white == perspective_is_white else 1
        p_idx = relative * 5 + ptype
        sq = orient(square, perspective_is_white)
        # Calculate index: KingSquare * 640 + PieceType * 64 + Square
        indices.append(np.uint16(k * 640 + p_idx * 64 + sq))
    return indices

def board_to_halfkp(fen):
    pieces, king_sq, us_is_white = parse_fen(fen, PIECE_CHAR_TO_TYPE)
    us_idx = halfkp_from_parsed(pieces, king_sq, us_is_white)
    them_idx = halfkp_from_parsed(pieces, king_sq, not us_is_white)
    return np.array(us_idx, dtype=np.uint16), np.array(them_idx, dtype=np.uint16)

In [14]:
split = dataset.train_test_split(test_size=0.2, seed=42)

In [15]:
train_val = split['train'].train_test_split(test_size=.2, seed=42)

In [8]:
MAX_PIECES = 32

def pad_indices(idx):
    padded = np.full(MAX_PIECES, 65535, dtype=np.uint16)
    if len(idx) > 0:
        n = min(len(idx), MAX_PIECES)
        padded[:n] = idx[:n]
    return padded




In [3]:
def create_example(fen, score):
    us_idx, them_idx = board_to_halfkp(fen)
    us_idx, them_idx = pad_indices(us_idx), pad_indices(them_idx)

    
    if CP_IS_WHITE_POV and fen.split()[1] == 'b':
        score = -score

    feature = {
        "us_idx": tf.train.Feature(bytes_list=tf.train.BytesList(value=[us_idx.tobytes()])),
        "them_idx": tf.train.Feature(bytes_list=tf.train.BytesList(value=[them_idx.tobytes()])),
        "score": tf.train.Feature(float_list=tf.train.FloatList(value=[np.tanh(score/400)])),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()

In [18]:
train_val

DatasetDict({
    train: Dataset({
        features: ['fen', 'depth', 'cp', 'mate'],
        num_rows: 53979868
    })
    test: Dataset({
        features: ['fen', 'depth', 'cp', 'mate'],
        num_rows: 13494968
    })
})

In [19]:
NUM_PROCESSES = os.cpu_count() or 16

def worker(args):
    split_name, process_id, dataset = args

    writer = tf.io.TFRecordWriter(
        f"{split_name}-{process_id:03d}.tfrecord"
    )

    invalid = 0

    for batch in dataset.iter(batch_size=2000):
        for fen, cp in zip(batch["fen"], batch["cp"]):
            try:
                writer.write(create_example(fen, cp))
            except ValueError:
                invalid += 1

    writer.close()

    return invalid


def process_split(split_key, split_name):
    shards = [
        train_val[split_key].shard(
            num_shards=NUM_PROCESSES,
            index=i,
            contiguous=True,
        )
        for i in range(NUM_PROCESSES)
    ]

    with mp.Pool(NUM_PROCESSES) as pool:
        invalid_counts = pool.map(
            worker,
            [
                (split_name, i, shards[i])
                for i in range(NUM_PROCESSES)
            ],
        )

    print(f"{split_name}: {sum(invalid_counts)} invalid positions")

In [22]:
process_split('train','train')

train: 0 invalid positions


In [23]:
process_split('test', 'val')

val: 0 invalid positions


In [5]:
def to_model_input(idx):
    padded = pad_indices(idx).astype(np.int32)
    padded[padded == 65535] = -1
    return padded[None, :]

In [6]:
feature_description = {
    "us_idx": tf.io.FixedLenFeature([], tf.string),
    "them_idx": tf.io.FixedLenFeature([], tf.string),
    "score": tf.io.FixedLenFeature([], tf.float32),
}

def parse_batch(serialized_examples):
    parsed = tf.io.parse_example(serialized_examples, feature_description)  
    us_idx = tf.io.decode_raw(parsed["us_idx"], tf.uint16)      
    them_idx = tf.io.decode_raw(parsed["them_idx"], tf.uint16)

    us_idx = tf.cast(us_idx, tf.int32)
    them_idx = tf.cast(them_idx, tf.int32)

    us_idx = tf.where(tf.equal(us_idx, 65535), -1, us_idx)
    them_idx = tf.where(tf.equal(them_idx, 65535), -1, them_idx)

    us_idx = tf.ensure_shape(us_idx, [None, MAX_PIECES])
    them_idx = tf.ensure_shape(them_idx, [None, MAX_PIECES])

    score = parsed["score"]
    return {"us_idx": us_idx, "them_idx": them_idx}, score

In [9]:
train_files = tf.io.gfile.glob("train-*.tfrecord")
val_files = tf.io.gfile.glob("val-*.tfrecord")

train_dataset = (
    tf.data.TFRecordDataset(train_files, num_parallel_reads=tf.data.AUTOTUNE)
    .shuffle(15000)
    .batch(8192, drop_remainder=True)  
    .map(parse_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.TFRecordDataset(val_files, num_parallel_reads=tf.data.AUTOTUNE)
    .batch(8192, drop_remainder=True)  
    .map(parse_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

In [10]:
EMBED_DIM = 256

def build_model():
    us_input = layers.Input(shape=(MAX_PIECES,), dtype=tf.int32, name="us_idx")
    them_input = layers.Input(shape=(MAX_PIECES,), dtype=tf.int32, name="them_idx")

    embed = layers.Embedding(input_dim=40961, output_dim=EMBED_DIM, name="embedding")

    us_shifted = us_input + 1
    them_shifted = them_input + 1
    mask_us = tf.keras.ops.cast(tf.keras.ops.not_equal(us_input, -1), "float32")[..., None]
    mask_them = tf.keras.ops.cast(tf.keras.ops.not_equal(them_input, -1), "float32")[..., None]

    us_vec = tf.keras.ops.sum(embed(us_shifted) * mask_us, axis=1)
    them_vec = tf.keras.ops.sum(embed(them_shifted) * mask_them, axis=1)

    x = layers.Concatenate(name="concatenate_1")([us_vec, them_vec])

    x = layers.Dense(256, activation='relu', name="dense")(x)
    x = layers.Dense(192, activation='relu', name="dense_1")(x)
    x = layers.Dense(128, activation='relu', name="dense_2")(x)
    out = layers.Dense(1, activation='tanh', dtype='float32', name="dense_3")(x)

    return tf.keras.Model(inputs={"us_idx": us_input, "them_idx": them_input}, outputs=out)

model = build_model()

In [11]:
CHECKPOINT_PATH = "/kaggle/working/best_chesseval.keras"  

if os.path.exists(CHECKPOINT_PATH):
    oldmodel = tf.keras.models.load_model(CHECKPOINT_PATH)

    for old_layer in oldmodel.layers:
        try:
            new_layer = model.get_layer(old_layer.name)
        except ValueError:
            continue

        old_weights = old_layer.get_weights()
        new_weights = new_layer.get_weights()

        if not old_weights:
            continue

        if all(
            a.shape == b.shape
            for a, b in zip(old_weights, new_weights)
        ):
            new_layer.set_weights(old_weights)
            print(f"{old_layer.name}: copied")
            continue

        if len(old_weights) == len(new_weights):
            copied = []

            for old_w, new_w in zip(old_weights, new_weights):
                w = new_w.copy()

                slices = tuple(
                    slice(0, min(a, b))
                    for a, b in zip(old_w.shape, new_w.shape)
                )

                w[slices] = old_w[slices]
                copied.append(w)

            new_layer.set_weights(copied)
            print(f"{old_layer.name}: partially copied")

    del oldmodel

    print("Loaded previous model weights.")

embedding: copied
dense: copied
dense_1: copied
dense_2: copied
dense_3: copied
Loaded previous model weights.


In [12]:
model.compile(
    optimizer='adam',
    loss="mean_absolute_error",
    metrics=['mse']
)

In [13]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [14]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_chesseval.keras",
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    save_freq='epoch'
)

In [15]:
model.fit(train_dataset, epochs=40, validation_data=val_dataset, callbacks=[early_stopping, checkpoint])

Epoch 1/40
      1/Unknown 4s 4s/step - loss: 0.1780 - mse: 0.0873

I0000 00:00:1786907897.850285     141 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   6589/Unknown 604s 91ms/step - loss: 0.1813 - mse: 0.0891

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


6589/6589 ━━━━━━━━━━━━━━━━━━━━ 662s 100ms/step - loss: 0.1818 - mse: 0.0896 - val_loss: 0.1934 - val_mse: 0.0989
Epoch 2/40
6589/6589 ━━━━━━━━━━━━━━━━━━━━ 656s 99ms/step - loss: 0.1806 - mse: 0.0889 - val_loss: 0.1940 - val_mse: 0.0999
Epoch 3/40
6589/6589 ━━━━━━━━━━━━━━━━━━━━ 659s 100ms/step - loss: 0.1797 - mse: 0.0884 - val_loss: 0.1939 - val_mse: 0.0998
Epoch 4/40
6589/6589 ━━━━━━━━━━━━━━━━━━━━ 660s 100ms/step - loss: 0.1789 - mse: 0.0879 - val_loss: 0.1946 - val_mse: 0.1010
Epoch 5/40
6589/6589 ━━━━━━━━━━━━━━━━━━━━ 664s 101ms/step - loss: 0.1783 - mse: 0.0876 - val_loss: 0.1945 - val_mse: 0.1002
Epoch 6/40
6589/6589 ━━━━━━━━━━━━━━━━━━━━ 657s 100ms/step - loss: 0.1777 - mse: 0.0872 - val_loss: 0.1946 - val_mse: 0.1016


In [34]:
model.save("chess.keras")

In [ ]:
with tf.io.TFRecordWriter("test.tfrecord") as writer:
    for batch in split['test'].iter(batch_size=2000):
        for fen, cp in zip(batch["fen"], batch["cp"]):
            writer.write(create_example(fen, cp))

In [ ]:
test_dataset = (
    tf.data.TFRecordDataset("test.tfrecord")
    .map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(512)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
model.evaluate(test_dataset)

In [ ]:
us_idx, them_idx = board_to_halfkp("4k3/8/8/2Q5/2q5/8/8/4K3 b - - 0 1")
us_idx = to_model_input(us_idx)
them_idx = to_model_input(them_idx)

prediction = model.predict({"us_idx": us_idx, "them_idx": them_idx})
print(prediction)


### 1. `parse_fen(fen)`

This function takes a FEN string as input and breaks it down into its constituent parts:

*   **Piece Placement**: It iterates through the FEN string to identify each piece, its type (pawn, knight, bishop, rook, queen, king), its color (white or black), and its square on the board.
*   **Turn**: It determines whose turn it is (`'w'` for white, `'b'` for black).

It returns a list of pieces with their squares and types, and the squares of the white and black kings.

### 2. `board_to_halfkp(fen)`

This is a crucial function that transforms the parsed FEN into a 'Half-KP' (Half-King-Piece) representation, which is a common way to encode chess positions for neural networks.

*   **`mirror_sq(square)`**: Helper function to mirror a square vertically, which is useful for maintaining a consistent perspective.
*   **`orient(square, perspective_is_white)`**: Orients the board based on the current player's perspective. If it's white's turn, the board is viewed from white's side; otherwise, it's mirrored for black's perspective.
*   **`halfkp_from_parsed(pieces, king_sq, perspective_is_white)`**: This function generates unique indices for each piece on the board, relative to the king's position and the current perspective. Each index incorporates the king's square, the piece type, its color relative to the current player, and the piece's square. This creates a compact numerical representation of the board state.

`board_to_halfkp` calls `parse_fen` and then uses `halfkp_from_parsed` twice: once for the current player's perspective (`us_idx`) and once for the opponent's perspective (`them_idx`).

### 3. `pad_indices(idx)`

*   **`MAX_PIECES = 32`**: A constant defining the maximum number of pieces that can be on a chess board (including both players' pieces).
*   This function takes the `us_idx` or `them_idx` arrays (which can have varying lengths depending on the number of pieces currently on the board) and pads them to a fixed length of `MAX_PIECES` using a sentinel value (`-1`). This ensures that all input arrays to the neural network have a consistent shape.

### 4. `create_example(fen, score)`

This function is responsible for assembling the pre-processed features into a `tf.train.Example`, which is the standard format for TensorFlow TFRecord files.

*   It calls `board_to_halfkp(fen)` to get the `us_idx` and `them_idx`.
*   It then calls `pad_indices()` on both sets of indices.
*   Finally, it creates a `tf.train.Example` protobuf, serializing the `us_idx` and `them_idx` as byte lists and the `score` (which is scaled using `np.tanh(score/400)`) as a float list. This `tf.train.Example` is then written to a TFRecord file for efficient loading during training.